<a href="https://colab.research.google.com/github/jackeywangxinyu/learnAngular/blob/master/HEM_JEPA_T4_representation_only_reviewed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HEM-JEPA on Colab T4  
**Hierarchical External-Memory JEPA with representation-only memory**

This notebook implements a **practical, single-session** version of the system we designed:

- base learner: **I-JEPA-style latent prediction**
- backbone: **ViT-like encoder**
- memory rule: **only representations are stored**
- hierarchy rule: **families are classified scientifically from measurable properties**
- evolution rule: the live model keeps training, while the memory bank tracks and filters the trajectory in representation space

## Hard constraints enforced here

1. **Drive / CPU memory bank only stores representations and metadata**
   - stored object: `z_t` and family centroids
   - stored metadata: time, energy, level, counts, confidence
   - never stored: raw weights, gradients, optimizer state, attention maps, hidden-state detail dumps

2. **The live model exists only inside the current Colab session**
   - this notebook does **not** save checkpoints
   - if the runtime disconnects, the representation bank survives, but the exact model weights do not

3. **Hierarchical levels are family-level properties**
   - not assigned by hand to single points
   - measured from: intrinsic dimension, energy, energy variance, repeatability, persistence, predictive closure

## What this notebook actually gives you

- a complete executable training loop
- auto batch-size search to push a T4 close to its usable memory ceiling
- hierarchical representation bank on Drive
- family-level scientific classification
- online state predictor over representation trajectories
- fast probe evaluation after training

## Recommended runtime modes

- **Smoke test**: `train_steps = 200`
- **Serious run**: `train_steps = 4000`
- **Long run**: `train_steps = 12000+` in one uninterrupted session

In [ ]:
# If you're on Colab, run this first.
# Minimal installs only. Colab T4 usually already has a CUDA-enabled torch build.
import importlib.util
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "numpy", "matplotlib", "scikit-learn", "tqdm", "psutil", "pyyaml"])

missing = [name for name in ("torch", "torchvision") if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(
        "Missing required packages: "
        + ", ".join(missing)
        + ". Colab T4 usually ships with torch/torchvision preinstalled. "
          "Install a CUDA-enabled torch/torchvision build matching your runtime before continuing."
    )


In [ ]:
import os
import io
import gc
import time
import json
import math
import copy
import psutil
import random
import pathlib
import warnings
from dataclasses import dataclass, asdict
from collections import deque, defaultdict
from typing import Dict, List, Tuple, Optional

import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

warnings.filterwarnings("ignore")

def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
amp_device_type = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print(f"physical VRAM: {props.total_memory / (1024**3):.2f} GB")


In [ ]:
# Optional Drive mount.
USE_DRIVE = True

if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=True)
        ROOT_DIR = pathlib.Path("/content/drive/MyDrive/hem_jepa_repr_only")
    except Exception as e:
        print("Drive mount skipped:", e)
        ROOT_DIR = pathlib.Path("/content/hem_jepa_repr_only")
else:
    ROOT_DIR = pathlib.Path("/content/hem_jepa_repr_only")

ROOT_DIR.mkdir(parents=True, exist_ok=True)
BANK_DIR = ROOT_DIR / "repr_bank"
BANK_DIR.mkdir(parents=True, exist_ok=True)
print("ROOT_DIR =", ROOT_DIR)

In [ ]:
@dataclass
class Config:
    dataset_name: str = "CIFAR100"
    data_root: str = "/content/data"
    image_size: int = 64
    patch_size: int = 4
    grid_size: int = 16
    num_classes: int = 100

    embed_dim: int = 512
    depth: int = 10
    num_heads: int = 8
    mlp_ratio: float = 4.0
    dropout: float = 0.0

    predictor_dim: int = 384
    predictor_depth: int = 6
    predictor_heads: int = 6

    state_dim: int = 128
    family_buffer_size: int = 96
    memory_max_points: int = 12000
    topk_pass_families: int = 6

    target_block_hw: int = 6
    num_target_blocks: int = 4

    train_steps: int = 4000
    warmup_steps: int = 200
    probe_train_size: int = 8000
    probe_test_size: int = 5000

    lr: float = 2e-4
    weight_decay: float = 0.05
    ema_start: float = 0.996
    ema_end: float = 0.9995
    grad_clip: float = 1.0

    state_pred_lr: float = 5e-4
    state_pred_updates_per_step: int = 1
    state_pred_start_after: int = 64

    lambda_state: float = 0.15
    lambda_work: float = 0.05

    batch_size: int = 0  # auto-tuned
    num_workers: int = 2
    pin_memory: bool = True

    target_peak_vram_gb: float = 14.2
    dryrun_steps: int = 3
    amp: bool = True
    track_attn_entropy: bool = False
    reclassify_all_every: int = 50

CFG = Config()
print(json.dumps(asdict(CFG), indent=2))


In [ ]:
# --------------------------
# Utilities
# --------------------------

def gpu_mem_gb() -> float:
    if not torch.cuda.is_available():
        return 0.0
    return torch.cuda.max_memory_allocated() / (1024**3)

def reset_gpu_peak() -> None:
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

def cosine_schedule(step: int, total_steps: int, start: float, end: float) -> float:
    if total_steps <= 1:
        return end
    t = min(max(step / (total_steps - 1), 0.0), 1.0)
    return end + 0.5 * (start - end) * (1.0 + math.cos(math.pi * t))

def block_positions(grid: int, block_hw: int, num_blocks: int, rng: random.Random) -> List[List[int]]:
    occupied = np.zeros((grid, grid), dtype=np.uint8)
    blocks = []
    tries = 0
    while len(blocks) < num_blocks and tries < 200:
        tries += 1
        r = rng.randint(0, grid - block_hw)
        c = rng.randint(0, grid - block_hw)
        if occupied[r:r+block_hw, c:c+block_hw].sum() > 0:
            continue
        occupied[r:r+block_hw, c:c+block_hw] = 1
        idxs = []
        for rr in range(r, r + block_hw):
            for cc in range(c, c + block_hw):
                idxs.append(rr * grid + cc)
        blocks.append(idxs)
    if len(blocks) < num_blocks:
        blocks = []
        stride = max(1, (grid - block_hw) // max(1, num_blocks - 1))
        for i in range(num_blocks):
            rr = min(i * stride, grid - block_hw)
            cc = min((i * stride) % max(1, grid - block_hw + 1), grid - block_hw)
            idxs = []
            for r in range(rr, rr + block_hw):
                for c in range(cc, cc + block_hw):
                    idxs.append(r * grid + c)
            blocks.append(idxs)
    return blocks

def sample_masks(batch_size: int, grid: int, block_hw: int, num_blocks: int, seed_offset: int = 0):
    total_tokens = grid * grid
    context_masks = torch.zeros(batch_size, total_tokens, dtype=torch.bool)
    target_masks = torch.zeros(batch_size, num_blocks, block_hw * block_hw, dtype=torch.long)
    all_target_flags = torch.zeros(batch_size, total_tokens, dtype=torch.bool)

    for b in range(batch_size):
        rng = random.Random(1234 + seed_offset * 100003 + b)
        blocks = block_positions(grid, block_hw, num_blocks, rng)
        used = []
        for i, block in enumerate(blocks):
            target_masks[b, i] = torch.tensor(block, dtype=torch.long)
            used.extend(block)
        all_target_flags[b, used] = True
        context_masks[b] = ~all_target_flags[b]
    return context_masks, target_masks, all_target_flags

def gather_tokens(x: torch.Tensor, idx: torch.Tensor) -> torch.Tensor:
    B, T, C = x.shape
    expanded = idx.unsqueeze(-1).expand(B, idx.shape[1], C)
    return x.gather(1, expanded)

def family_intrinsic_dim(vectors: np.ndarray) -> float:
    n, d = vectors.shape
    if n < 4:
        return 1.0
    x = vectors - vectors.mean(axis=0, keepdims=True)
    cov = x.T @ x / max(n - 1, 1)
    eigvals = np.linalg.eigvalsh(cov)
    eigvals = np.clip(eigvals, 1e-12, None)
    pr = (eigvals.sum() ** 2) / (np.square(eigvals).sum() + 1e-12)
    return float(pr)

def family_knn_dim(vectors: np.ndarray, k: int = 5) -> float:
    n, d = vectors.shape
    if n < k + 2:
        return 1.0
    x = vectors
    dist = np.sqrt(((x[:, None, :] - x[None, :, :]) ** 2).sum(-1) + 1e-12)
    np.fill_diagonal(dist, np.inf)
    nn = np.sort(dist, axis=1)[:, :k]
    rk = nn[:, -1]
    logs = np.log((rk[:, None] + 1e-12) / (nn[:, :-1] + 1e-12))
    est = (k - 1) / (logs.sum(axis=1) + 1e-12)
    est = np.clip(est, 1.0, d)
    return float(np.median(est))

def robust_quantiles(values: List[float], fallback: Tuple[float, float, float] = (0.2, 0.5, 0.8)):
    if len(values) < 8:
        return fallback
    arr = np.asarray(values, dtype=np.float32)
    return tuple(np.quantile(arr, [0.2, 0.5, 0.8]).tolist())

In [ ]:
# --------------------------
# Model
# --------------------------

class TransformerBlock(nn.Module):
    def __init__(self, dim: int, num_heads: int, mlp_ratio: float = 4.0, dropout: float = 0.0, track_attn_entropy: bool = False):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, num_heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.track_attn_entropy = track_attn_entropy
        hidden = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(dim, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, dim),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor, key_padding_mask: Optional[torch.Tensor] = None):
        h = self.norm1(x)
        attn_out, attn_weights = self.attn(
            h, h, h,
            key_padding_mask=key_padding_mask,
            need_weights=self.track_attn_entropy,
            average_attn_weights=False,
        )
        x = x + attn_out
        h2 = self.norm2(x)
        x = x + self.mlp(h2)

        with torch.no_grad():
            token_norm = x.detach().norm(dim=-1).mean(dim=-1)
            if attn_weights is None:
                attn_entropy = torch.zeros(x.size(0), device=x.device, dtype=torch.float32)
            else:
                p = attn_weights.float().clamp_min(1e-8)
                attn_entropy = (-p * p.log()).sum(dim=-1).mean(dim=(-1, -2))
        stats = {
            "token_norm_mean": token_norm,
            "attn_entropy": attn_entropy,
            "act_mean": x.mean(dim=(1, 2)).detach(),
            "act_std": x.std(dim=(1, 2)).detach(),
        }
        return x, stats


class ViTEncoder(nn.Module):
    def __init__(self, image_size: int, patch_size: int, in_chans: int, dim: int, depth: int, num_heads: int, mlp_ratio: float, track_attn_entropy: bool = False):
        super().__init__()
        assert image_size % patch_size == 0
        self.image_size = image_size
        self.patch_size = patch_size
        self.grid = image_size // patch_size
        self.num_tokens = self.grid * self.grid
        self.dim = dim

        self.patch_embed = nn.Conv2d(in_chans, dim, kernel_size=patch_size, stride=patch_size)
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_tokens, dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        self.blocks = nn.ModuleList([
            TransformerBlock(dim, num_heads, mlp_ratio, track_attn_entropy=track_attn_entropy) for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(dim)

    def forward(self, x: torch.Tensor, visible_mask: Optional[torch.Tensor] = None):
        x = self.patch_embed(x).flatten(2).transpose(1, 2)
        x = x + self.pos_embed

        key_padding_mask = None
        if visible_mask is not None:
            key_padding_mask = ~visible_mask
            x = x * visible_mask.unsqueeze(-1)

        layer_stats = []
        for blk in self.blocks:
            x, stats = blk(x, key_padding_mask=key_padding_mask)
            layer_stats.append(stats)
        x = self.norm(x)
        pooled = x.mean(dim=1)
        return x, pooled, layer_stats


class JEPAImagePredictor(nn.Module):
    def __init__(self, in_dim: int, pred_dim: int, depth: int, heads: int):
        super().__init__()
        self.ctx_proj = nn.Linear(in_dim, pred_dim)
        self.query_embed = nn.Parameter(torch.randn(1, 1, pred_dim) * 0.02)
        self.pos_proj = nn.Linear(2, pred_dim)
        self.blocks = nn.ModuleList([
            TransformerBlock(pred_dim, heads, mlp_ratio=4.0) for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(pred_dim)
        self.out = nn.Linear(pred_dim, in_dim)

    def forward(self, context_tokens: torch.Tensor, target_positions_xy: torch.Tensor):
        ctx = self.ctx_proj(context_tokens)
        q = self.query_embed.expand(context_tokens.size(0), target_positions_xy.size(1), -1)
        q = q + self.pos_proj(target_positions_xy)
        x = torch.cat([ctx, q], dim=1)
        for blk in self.blocks:
            x, _ = blk(x, key_padding_mask=None)
        x = self.norm(x)
        pred = self.out(x[:, -target_positions_xy.size(1):])
        return pred


class StatePredictor(nn.Module):
    def __init__(self, state_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim + 4, 256),
            nn.GELU(),
            nn.Linear(256, 256),
            nn.GELU(),
            nn.Linear(256, state_dim),
        )

    def forward(self, z: torch.Tensor, aux: torch.Tensor) -> torch.Tensor:
        return F.normalize(self.net(torch.cat([z, aux], dim=-1)), dim=-1)


In [ ]:
# --------------------------
# Representation-only summarizer
# --------------------------

class StateSummarizer:
    def __init__(self, model: ViTEncoder, out_dim: int = 128, seed: int = 123):
        self.out_dim = out_dim
        self.seed = seed
        self._proj = None
        self._build_projection(model)

    def _build_projection(self, model: ViTEncoder):
        rng = np.random.default_rng(self.seed)
        dummy = self._summary_vector(model, None, None, build_only=True)
        in_dim = dummy.shape[0]
        proj = rng.standard_normal((in_dim, self.out_dim)).astype(np.float32)
        proj /= np.linalg.norm(proj, axis=0, keepdims=True) + 1e-8
        self._proj = torch.tensor(proj, dtype=torch.float32, device=device)

    def _summary_vector(self, model: ViTEncoder, pooled: Optional[torch.Tensor], layer_stats: Optional[List[Dict]], build_only: bool = False) -> torch.Tensor:
        feats = []
        pe = model.patch_embed.weight
        feats += [pe.mean().detach().float(), pe.std().detach().float(), pe.norm().detach().float()]
        pos = model.pos_embed
        feats += [pos.mean().detach().float(), pos.std().detach().float()]

        for blk in model.blocks:
            qkv_like = blk.attn.in_proj_weight
            outp = blk.attn.out_proj.weight
            fc1 = blk.mlp[0].weight
            fc2 = blk.mlp[3].weight
            ln1 = blk.norm1.weight
            ln2 = blk.norm2.weight
            for w in (qkv_like, outp, fc1, fc2):
                feats += [w.mean().detach().float(), w.std().detach().float(), w.norm().detach().float()]
            for ln in (ln1, ln2):
                feats += [ln.mean().detach().float(), ln.std().detach().float()]

        if build_only or pooled is None or layer_stats is None:
            n_blocks = len(model.blocks)
            behavior_dim = 2 + n_blocks * 4
            feats += [torch.tensor(0.0, device=device) for _ in range(behavior_dim)]
        else:
            feats += [pooled.mean().detach().float(), pooled.std().detach().float()]
            for st in layer_stats:
                feats += [
                    st["token_norm_mean"].mean().detach().float(),
                    st["attn_entropy"].mean().detach().float(),
                    st["act_mean"].mean().detach().float(),
                    st["act_std"].mean().detach().float(),
                ]

        return torch.stack(feats)

    @torch.no_grad()
    def __call__(self, model: ViTEncoder, pooled: torch.Tensor, layer_stats: List[Dict]) -> torch.Tensor:
        s = self._summary_vector(model, pooled, layer_stats)
        z = s @ self._proj
        return F.normalize(z, dim=0)

In [ ]:
# --------------------------
# Representation family bank
# --------------------------

LEVEL_NAMES = {
    0: "L0 catastrophic",
    1: "L1 structural_error",
    2: "L2 detail_error",
    3: "L3 feasible_boundary",
    4: "L4 working_attractor",
}

class Family:
    def __init__(self, family_id: int, z: np.ndarray, energy: float, step: int):
        self.family_id = family_id
        self.centroid = z.astype(np.float32)
        self.vectors = deque([z.astype(np.float32)], maxlen=CFG.family_buffer_size)
        self.energies = deque([float(energy)], maxlen=CFG.family_buffer_size)
        self.steps = deque([int(step)], maxlen=CFG.family_buffer_size)
        self.pred_errors = deque([], maxlen=CFG.family_buffer_size)
        self.count = 1
        self.first_step = step
        self.last_step = step
        self.level = 3
        self.confidence = 0.5
        self._stats_cache = None
        self._dirty = True

    def add(self, z: np.ndarray, energy: float, step: int, pred_error: Optional[float] = None):
        self.vectors.append(z.astype(np.float32))
        self.energies.append(float(energy))
        self.steps.append(int(step))
        if pred_error is not None:
            self.pred_errors.append(float(pred_error))
        self.count += 1
        self.last_step = step
        alpha = min(0.2, 1.0 / max(self.count, 1))
        self.centroid = (1 - alpha) * self.centroid + alpha * z.astype(np.float32)
        self._dirty = True

    def stats(self):
        if (not self._dirty) and (self._stats_cache is not None):
            return self._stats_cache
        vecs = np.stack(list(self.vectors), axis=0)
        energies = np.asarray(list(self.energies), dtype=np.float32)
        d_pr = family_intrinsic_dim(vecs)
        d_nn = family_knn_dim(vecs, k=min(5, max(2, len(vecs) - 1)))
        intrinsic_dim = float(np.median([d_pr, d_nn]))
        mean_energy = float(energies.mean())
        std_energy = float(energies.std())
        repeatability = float(np.log1p(self.count))
        persistence = float(self.last_step - self.first_step + 1)
        closure = 0.5 if len(self.pred_errors) == 0 else float(1.0 / (1.0 + np.mean(list(self.pred_errors))))
        self._stats_cache = {
            "intrinsic_dim": intrinsic_dim,
            "mean_energy": mean_energy,
            "std_energy": std_energy,
            "repeatability": repeatability,
            "persistence": persistence,
            "closure": closure,
        }
        self._dirty = False
        return self._stats_cache

class RepresentationBank:
    def __init__(self, bank_dir: pathlib.Path, state_dim: int):
        self.bank_dir = bank_dir
        self.state_dim = state_dim
        self.families: Dict[int, Family] = {}
        self.next_family_id = 0
        self.history = deque([], maxlen=CFG.memory_max_points)
        self.recent_z = deque([], maxlen=512)
        self.family_assign_threshold = 0.14

    def _cosine_dist(self, a: np.ndarray, b: np.ndarray) -> float:
        return float(1.0 - np.dot(a, b) / ((np.linalg.norm(a) * np.linalg.norm(b)) + 1e-8))

    def _assign_family(self, z: np.ndarray) -> int:
        if len(self.families) == 0:
            fid = self.next_family_id
            self.next_family_id += 1
            return fid
        dists = {fid: self._cosine_dist(z, fam.centroid) for fid, fam in self.families.items()}
        fid_best, dist_best = min(dists.items(), key=lambda kv: kv[1])
        if dist_best > self.family_assign_threshold:
            fid = self.next_family_id
            self.next_family_id += 1
            return fid
        return fid_best

    def global_quantiles(self):
        if len(self.families) < 5:
            return None
        dims, means, closures, reps, pers = [], [], [], [], []
        for fam in self.families.values():
            st = fam.stats()
            dims.append(st["intrinsic_dim"])
            means.append(st["mean_energy"])
            closures.append(st["closure"])
            reps.append(st["repeatability"])
            pers.append(st["persistence"])
        return {
            "dim": robust_quantiles(dims),
            "energy": robust_quantiles(means),
            "closure": robust_quantiles(closures),
            "repeatability": robust_quantiles(reps),
            "persistence": robust_quantiles(pers),
        }

    def classify_family(self, fam: Family):
        st = fam.stats()
        qs = self.global_quantiles()
        if qs is None:
            fam.level = 3
            fam.confidence = 0.5
            return fam.level, fam.confidence

        d20, d50, d80 = qs["dim"]
        e20, e50, e80 = qs["energy"]
        c20, c50, c80 = qs["closure"]
        r20, r50, r80 = qs["repeatability"]
        p20, p50, p80 = qs["persistence"]

        d = st["intrinsic_dim"]
        e = st["mean_energy"]
        c = st["closure"]
        r = st["repeatability"]
        p = st["persistence"]

        if d <= d20 and e >= e80 and p <= p50:
            fam.level = 0; fam.confidence = 0.90
        elif d <= d50 and e >= e80:
            fam.level = 1; fam.confidence = 0.80
        elif d >= d50 and e >= e50:
            fam.level = 2; fam.confidence = 0.70
        elif e <= e20 and c >= c80 and r >= r50 and p >= p50:
            fam.level = 4; fam.confidence = 0.95
        else:
            fam.level = 3; fam.confidence = 0.65
        return fam.level, fam.confidence

    def add(self, z: torch.Tensor, energy: float, step: int, pred_error: Optional[float] = None):
        z_np = z.detach().float().cpu().numpy()
        fid = self._assign_family(z_np)
        if fid not in self.families:
            self.families[fid] = Family(fid, z_np, energy, step)
        else:
            self.families[fid].add(z_np, energy, step, pred_error=pred_error)

        self.classify_family(self.families[fid])
        if CFG.reclassify_all_every > 0 and (step + 1) % CFG.reclassify_all_every == 0:
            for other_fam in self.families.values():
                if other_fam.family_id != fid:
                    self.classify_family(other_fam)

        fam = self.families[fid]
        self.history.append({
            "step": int(step),
            "family_id": int(fid),
            "energy": float(energy),
            "level": int(fam.level),
            "confidence": float(fam.confidence),
            "z": z_np.astype(np.float32),
        })
        self.recent_z.append(z_np.astype(np.float32))
        return fid, fam.level

    def working_representation(self) -> Optional[torch.Tensor]:
        pass_fams = []
        for fam in self.families.values():
            if fam.level in (3, 4):
                st = fam.stats()
                score = -st["mean_energy"] + 0.25 * st["closure"] + 0.05 * math.log1p(fam.count)
                pass_fams.append((score, fam))
        if len(pass_fams) == 0:
            return None
        pass_fams = sorted(pass_fams, key=lambda x: x[0], reverse=True)[:CFG.topk_pass_families]
        scores = torch.tensor([s for s, _ in pass_fams], dtype=torch.float32, device=device)
        weights = torch.softmax(scores, dim=0)
        vecs = torch.tensor(np.stack([fam.centroid for _, fam in pass_fams]), dtype=torch.float32, device=device)
        z_star = (weights.unsqueeze(-1) * vecs).sum(dim=0)
        return F.normalize(z_star, dim=0)

    def save_repr_bank(self, name: str = "repr_bank_latest.npz"):
        path = self.bank_dir / name
        if len(self.history) == 0:
            return path
        steps = np.asarray([h["step"] for h in self.history], dtype=np.int32)
        family_ids = np.asarray([h["family_id"] for h in self.history], dtype=np.int32)
        levels = np.asarray([h["level"] for h in self.history], dtype=np.int32)
        energies = np.asarray([h["energy"] for h in self.history], dtype=np.float32)
        confidences = np.asarray([h["confidence"] for h in self.history], dtype=np.float32)
        zs = np.stack([h["z"] for h in self.history]).astype(np.float32)
        np.savez_compressed(path, steps=steps, family_ids=family_ids, levels=levels, energies=energies, confidences=confidences, z=zs)
        return path

    def summary_frame(self):
        rows = []
        for fid, fam in self.families.items():
            st = fam.stats()
            rows.append({
                "family_id": fid,
                "count": fam.count,
                "level": LEVEL_NAMES[fam.level],
                "confidence": round(fam.confidence, 3),
                "intrinsic_dim": round(st["intrinsic_dim"], 3),
                "mean_energy": round(st["mean_energy"], 4),
                "std_energy": round(st["std_energy"], 4),
                "closure": round(st["closure"], 4),
                "persistence": round(st["persistence"], 1),
            })
        return sorted(rows, key=lambda r: (-r["count"], r["mean_energy"]))


In [ ]:
# --------------------------
# Data
# --------------------------

def build_datasets(cfg: Config):
    if cfg.dataset_name.upper() == "CIFAR100":
        mean = (0.5071, 0.4867, 0.4408)
        std = (0.2675, 0.2565, 0.2761)
        cfg.num_classes = 100
    else:
        mean = (0.4914, 0.4822, 0.4465)
        std = (0.2470, 0.2435, 0.2616)
        cfg.num_classes = 10

    train_tf = transforms.Compose([
        transforms.Resize((cfg.image_size, cfg.image_size)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    test_tf = transforms.Compose([
        transforms.Resize((cfg.image_size, cfg.image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])

    if cfg.dataset_name.upper() == "CIFAR100":
        train_ds = datasets.CIFAR100(root=cfg.data_root, train=True, download=True, transform=train_tf)
        test_ds = datasets.CIFAR100(root=cfg.data_root, train=False, download=True, transform=test_tf)
    else:
        train_ds = datasets.CIFAR10(root=cfg.data_root, train=True, download=True, transform=train_tf)
        test_ds = datasets.CIFAR10(root=cfg.data_root, train=False, download=True, transform=test_tf)
    return train_ds, test_ds

train_ds, test_ds = build_datasets(CFG)
print(len(train_ds), len(test_ds))

In [ ]:
# --------------------------
# Auto batch tuner
# --------------------------

def make_models(cfg: Config):
    online = ViTEncoder(cfg.image_size, cfg.patch_size, 3, cfg.embed_dim, cfg.depth, cfg.num_heads, cfg.mlp_ratio, track_attn_entropy=cfg.track_attn_entropy).to(device)
    target = copy.deepcopy(online).to(device)
    for p in target.parameters():
        p.requires_grad = False
    predictor = JEPAImagePredictor(cfg.embed_dim, cfg.predictor_dim, cfg.predictor_depth, cfg.predictor_heads).to(device)
    state_pred = StatePredictor(cfg.state_dim).to(device)
    return online, target, predictor, state_pred

@torch.no_grad()
def positions_from_indices(idx: torch.Tensor, grid: int) -> torch.Tensor:
    r = (idx // grid).float() / max(1, grid - 1)
    c = (idx % grid).float() / max(1, grid - 1)
    return torch.stack([r, c], dim=-1)

def dryrun_peak_memory(cfg: Config, batch_size: int) -> float:
    online, target, predictor, _ = make_models(cfg)
    online.train(); predictor.train()
    opt = torch.optim.AdamW(list(online.parameters()) + list(predictor.parameters()), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scaler = torch.amp.GradScaler(amp_device_type, enabled=(cfg.amp and torch.cuda.is_available()))
    reset_gpu_peak()

    try:
        for step in range(cfg.dryrun_steps):
            x = torch.randn(batch_size, 3, cfg.image_size, cfg.image_size, device=device)
            context_mask, target_masks, _ = sample_masks(batch_size, cfg.grid_size, cfg.target_block_hw, cfg.num_target_blocks, seed_offset=step)
            context_mask = context_mask.to(device)
            target_masks = target_masks.to(device)
            flat_targets = target_masks.reshape(batch_size, -1)
            target_pos = positions_from_indices(flat_targets, cfg.grid_size)

            with torch.amp.autocast(amp_device_type, enabled=(cfg.amp and torch.cuda.is_available())):
                ctx_tokens, _, _ = online(x, visible_mask=context_mask)
                tgt_tokens, _, _ = target(x, visible_mask=None)
                pred = predictor(ctx_tokens, target_pos)
                tgt = gather_tokens(tgt_tokens, flat_targets)
                loss = F.mse_loss(pred, tgt)

            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
        peak = gpu_mem_gb()
    finally:
        del online, target, predictor, opt, scaler
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return peak

def autotune_batch_size(cfg: Config) -> int:
    if not torch.cuda.is_available():
        return 16
    candidates = [32, 48, 64, 80, 96, 112, 128, 160, 192, 224]
    best_bs, best_peak = candidates[0], 0.0
    for bs in candidates:
        try:
            peak = dryrun_peak_memory(cfg, bs)
            print(f"batch={bs:>3d} -> peak={peak:.2f} GB")
            if peak <= cfg.target_peak_vram_gb:
                best_bs, best_peak = bs, peak
            else:
                break
        except RuntimeError:
            print(f"batch={bs:>3d} -> OOM / stop")
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            break
    print(f"chosen batch size: {best_bs}, expected peak ~{best_peak:.2f} GB")
    return best_bs

if CFG.batch_size <= 0:
    CFG.batch_size = autotune_batch_size(CFG)

print("final batch_size =", CFG.batch_size)


In [ ]:
# --------------------------
# Training objects
# --------------------------

online_encoder, target_encoder, image_predictor, state_predictor = make_models(CFG)
work_projector = nn.Linear(CFG.embed_dim, CFG.state_dim, bias=False).to(device)
nn.init.orthogonal_(work_projector.weight)
state_summarizer = StateSummarizer(online_encoder, out_dim=CFG.state_dim)
repr_bank = RepresentationBank(BANK_DIR, CFG.state_dim)

train_loader = DataLoader(
    train_ds,
    batch_size=CFG.batch_size,
    shuffle=True,
    num_workers=CFG.num_workers,
    pin_memory=CFG.pin_memory,
    drop_last=True,
)

probe_train_subset = Subset(train_ds, list(range(min(CFG.probe_train_size, len(train_ds)))))
probe_test_subset = Subset(test_ds, list(range(min(CFG.probe_test_size, len(test_ds)))))

probe_train_loader = DataLoader(probe_train_subset, batch_size=CFG.batch_size, shuffle=False, num_workers=CFG.num_workers)
probe_test_loader = DataLoader(probe_test_subset, batch_size=CFG.batch_size, shuffle=False, num_workers=CFG.num_workers)

main_optim = torch.optim.AdamW(
    list(online_encoder.parameters()) + list(image_predictor.parameters()) + list(work_projector.parameters()),
    lr=CFG.lr,
    weight_decay=CFG.weight_decay,
)
state_optim = torch.optim.AdamW(state_predictor.parameters(), lr=CFG.state_pred_lr, weight_decay=1e-4)

main_scaler = torch.amp.GradScaler(amp_device_type, enabled=(CFG.amp and torch.cuda.is_available()))
state_scaler = torch.amp.GradScaler(amp_device_type, enabled=(CFG.amp and torch.cuda.is_available()))

print("models ready")


In [ ]:
# --------------------------
# Core training logic
# --------------------------

def update_ema(online: nn.Module, target: nn.Module, m: float):
    with torch.no_grad():
        for p_o, p_t in zip(online.parameters(), target.parameters()):
            p_t.data.mul_(m).add_(p_o.data, alpha=(1.0 - m))

@torch.no_grad()
def encoder_features(model: ViTEncoder, loader: DataLoader, max_batches: Optional[int] = None):
    model.eval()
    feats, labels = [], []
    for i, (x, y) in enumerate(loader):
        if max_batches is not None and i >= max_batches:
            break
        x = x.to(device, non_blocking=True)
        _, pooled, _ = model(x, visible_mask=None)
        feats.append(pooled.detach().cpu().numpy())
        labels.append(y.numpy())
    return np.concatenate(feats, axis=0), np.concatenate(labels, axis=0)

def fast_linear_probe(model: ViTEncoder):
    X_train, y_train = encoder_features(model, probe_train_loader)
    X_test, y_test = encoder_features(model, probe_test_loader)
    clf = LogisticRegression(max_iter=200, n_jobs=1, verbose=0)
    clf.fit(X_train, y_train)
    pred = clf.predict(X_test)
    return accuracy_score(y_test, pred)

history = {
    "img_loss": [], "state_loss": [], "energy": [], "lr_scale": [], "ema_m": [], "peak_vram_gb": [], "probe_acc": [],
}

z_sequence = deque([], maxlen=2048)
aux_sequence = deque([], maxlen=2048)
next_lr_scale = 1.0
next_ema_delta = 0.0

train_iter = iter(train_loader)
start_time = time.time()

for step in tqdm(range(CFG.train_steps), desc="training"):
    try:
        x, _ = next(train_iter)
    except StopIteration:
        train_iter = iter(train_loader)
        x, _ = next(train_iter)
    x = x.to(device, non_blocking=True)

    for g in main_optim.param_groups:
        g["lr"] = CFG.lr * next_lr_scale

    context_mask, target_masks, _ = sample_masks(x.size(0), CFG.grid_size, CFG.target_block_hw, CFG.num_target_blocks, seed_offset=step)
    context_mask = context_mask.to(device)
    target_masks = target_masks.to(device)
    flat_targets = target_masks.reshape(x.size(0), -1)
    target_pos = positions_from_indices(flat_targets, CFG.grid_size)

    reset_gpu_peak()
    online_encoder.train(); image_predictor.train(); work_projector.train()

    with torch.amp.autocast(amp_device_type, enabled=(CFG.amp and torch.cuda.is_available())):
        ctx_tokens, pooled_ctx, layer_stats = online_encoder(x, visible_mask=context_mask)
        with torch.no_grad():
            tgt_tokens, tgt_pooled, tgt_layer_stats = target_encoder(x, visible_mask=None)
        pred_tokens = image_predictor(ctx_tokens, target_pos)
        target_tokens = gather_tokens(tgt_tokens, flat_targets)
        img_loss = F.mse_loss(pred_tokens, target_tokens)

        work_z = repr_bank.working_representation()
        if work_z is None:
            work_loss = torch.tensor(0.0, device=device)
        else:
            pooled_state = F.normalize(work_projector(pooled_ctx.mean(dim=0)), dim=0)
            work_loss = 1.0 - torch.clamp((pooled_state * work_z).sum(), -1.0, 1.0)

        total_main_loss = img_loss + CFG.lambda_work * work_loss

    main_optim.zero_grad(set_to_none=True)
    main_scaler.scale(total_main_loss).backward()
    main_scaler.unscale_(main_optim)
    torch.nn.utils.clip_grad_norm_(list(online_encoder.parameters()) + list(image_predictor.parameters()) + list(work_projector.parameters()), CFG.grad_clip)
    main_scaler.step(main_optim)
    main_scaler.update()

    ema_m = cosine_schedule(step, CFG.train_steps, CFG.ema_start, CFG.ema_end)
    ema_m = min(max(ema_m + next_ema_delta, 0.99), 0.9999)
    update_ema(online_encoder, target_encoder, ema_m)

    with torch.no_grad():
        # Use the EMA target encoder's unmasked behavior as a cheap proxy for current full-image behavior.
        # This avoids a redundant second online forward on every step.
        z_t = state_summarizer(online_encoder, tgt_pooled, tgt_layer_stats)

    z_np = z_t.detach().cpu().numpy()
    if len(repr_bank.recent_z) > 8:
        recent = np.stack(list(repr_bank.recent_z))
        center = recent.mean(axis=0)
        drift = 1.0 - float(np.dot(z_np, center) / ((np.linalg.norm(z_np) * np.linalg.norm(center)) + 1e-8))
    else:
        drift = 0.0

    energy = 0.70 * float(img_loss.detach().cpu().item()) + 0.20 * float(work_loss.detach().cpu().item()) + 0.10 * float(drift)
    aux_t = torch.tensor([
        float(img_loss.detach().cpu().item()),
        float(work_loss.detach().cpu().item()),
        float(drift),
        float(next_lr_scale),
    ], dtype=torch.float32, device=device)

    state_loss_value = 0.0
    if len(z_sequence) >= CFG.state_pred_start_after:
        for _ in range(CFG.state_pred_updates_per_step):
            idx = random.randint(0, len(z_sequence) - 2)
            z_now = z_sequence[idx].to(device)
            z_next = z_sequence[idx + 1].to(device)
            aux_now = aux_sequence[idx].to(device)
            with torch.amp.autocast(amp_device_type, enabled=(CFG.amp and torch.cuda.is_available())):
                z_hat = state_predictor(z_now.unsqueeze(0), aux_now.unsqueeze(0)).squeeze(0)
                state_loss = F.mse_loss(z_hat, z_next)
            state_optim.zero_grad(set_to_none=True)
            state_scaler.scale(state_loss).backward()
            state_scaler.step(state_optim)
            state_scaler.update()
            state_loss_value += float(state_loss.detach().cpu().item())
        state_loss_value /= CFG.state_pred_updates_per_step

    pred_error_for_bank = None
    if len(z_sequence) >= 1:
        with torch.no_grad():
            z_prev = z_sequence[-1].to(device)
            aux_prev = aux_sequence[-1].to(device)
            z_hat_prev = state_predictor(z_prev.unsqueeze(0), aux_prev.unsqueeze(0)).squeeze(0)
            pred_error_for_bank = float(F.mse_loss(z_hat_prev, z_t).detach().cpu().item())

    family_id, level = repr_bank.add(z_t, energy=energy, step=step, pred_error=pred_error_for_bank)
    z_sequence.append(z_t.detach().cpu())
    aux_sequence.append(aux_t.detach().cpu())

    work_z = repr_bank.working_representation()
    pass_sim = 0.0 if work_z is None else float((z_t * work_z).sum().detach().cpu().item())
    fam = repr_bank.families[family_id]
    if fam.level in (0, 1):
        next_lr_scale, next_ema_delta = 0.70, 0.002
    elif fam.level == 2:
        next_lr_scale, next_ema_delta = 0.85, 0.001
    elif fam.level == 4 and pass_sim > 0.85:
        next_lr_scale, next_ema_delta = 1.05, -0.0003
    else:
        next_lr_scale, next_ema_delta = 1.0, 0.0

    history["img_loss"].append(float(img_loss.detach().cpu().item()))
    history["state_loss"].append(float(state_loss_value))
    history["energy"].append(float(energy))
    history["lr_scale"].append(float(next_lr_scale))
    history["ema_m"].append(float(ema_m))
    history["peak_vram_gb"].append(float(gpu_mem_gb()))

    if (step + 1) % 250 == 0:
        bank_path = repr_bank.save_repr_bank()
        print(f"\nstep={step+1} saved representation bank -> {bank_path}")

    if (step + 1) % 100 == 0:
        elapsed = time.time() - start_time
        sec_per_step = elapsed / (step + 1)
        eta_hr = sec_per_step * (CFG.train_steps - step - 1) / 3600.0
        print(
            f"\nstep={step+1}/{CFG.train_steps} | img_loss={history['img_loss'][-1]:.4f} | state_loss={history['state_loss'][-1]:.4f} | energy={history['energy'][-1]:.4f} | peak_vram={history['peak_vram_gb'][-1]:.2f} GB | ETA={eta_hr:.2f} h"
        )

final_bank_path = repr_bank.save_repr_bank("repr_bank_final.npz")
print("final representation bank:", final_bank_path)


In [ ]:
# --------------------------
# Evaluation
# --------------------------

probe_acc = fast_linear_probe(target_encoder)
history["probe_acc"].append(float(probe_acc))
print(f"Fast linear probe accuracy: {probe_acc:.4f}")

In [ ]:
# --------------------------
# Visualizations
# --------------------------

plt.figure(figsize=(14, 4))
plt.subplot(1, 3, 1)
plt.plot(history["img_loss"], label="img_loss")
plt.plot(history["state_loss"], label="state_loss", alpha=0.8)
plt.legend(); plt.title("Loss curves")

plt.subplot(1, 3, 2)
plt.plot(history["energy"], label="energy")
plt.plot(history["lr_scale"], label="next_lr_scale")
plt.legend(); plt.title("Energy and steering")

plt.subplot(1, 3, 3)
plt.plot(history["peak_vram_gb"], label="peak_vram_gb")
plt.axhline(CFG.target_peak_vram_gb, linestyle="--", label="target")
plt.legend(); plt.title("Peak VRAM")
plt.tight_layout(); plt.show()

rows = repr_bank.summary_frame()
print("Top families:")
for row in rows[:20]:
    print(row)

In [ ]:
# --------------------------
# Optional: 2D visualization of family centroids
# --------------------------

if len(repr_bank.families) >= 3:
    cents = np.stack([fam.centroid for fam in repr_bank.families.values()])
    levels = [fam.level for fam in repr_bank.families.values()]
    family_ids = [fam.family_id for fam in repr_bank.families.values()]

    pca = PCA(n_components=2)
    xy = pca.fit_transform(cents)

    plt.figure(figsize=(7, 6))
    for lvl in sorted(set(levels)):
        idx = [i for i, x in enumerate(levels) if x == lvl]
        plt.scatter(xy[idx, 0], xy[idx, 1], label=LEVEL_NAMES[lvl], s=60)
    for i, fid in enumerate(family_ids):
        plt.text(xy[i, 0], xy[i, 1], str(fid), fontsize=8)
    plt.legend(); plt.title("Family centroids in PCA space")
    plt.show()
else:
    print("Need at least 3 families for centroid PCA plot.")

## Runtime interpretation

The notebook prints a live ETA every 100 steps, which is more trustworthy than any fixed estimate because Colab hosts vary.

### Practical expectations on a Colab T4

This implementation intentionally uses:

- 64×64 inputs
- a ViT-like encoder with 16×16 token grid
- 4 target blocks of size 6×6
- a predictor branch
- online representation extraction
- online family-bank updates

That makes it **substantially heavier** than toy CIFAR notebooks, but still realistic for a T4.

Typical ranges you should expect:

- **200-step smoke test**: about 10–20 minutes
- **4000-step serious run**: about 3–6 hours
- **12000-step long run**: about 9–18 hours

The exact number depends on the batch size chosen by the auto-tuner, Colab host load, and whether your runtime gets a fast or slow CPU feeder.

## What this system can become under long uninterrupted training

### Best realistic outcome in this notebook
Within a single long T4 session, the model can evolve toward:

- a **stable low-energy representation trajectory**
- a **cleaner pass-family structure**
- a **working-attractor family** that becomes more repeatable over time
- better latent organization than plain reconstruction-style toy training
- modest downstream gains on CIFAR-style probe tasks

### What it will not magically become
This notebook will **not** turn a T4 into a large-scale world model.

Because of your hard constraint that only representations may be stored:

- the session cannot truly resume from exact old weights
- multi-day continual evolution across disconnected Colab sessions is not exact
- the system remembers the **shape of its trajectory in representation space**, not the full live parameter state

### Strongest form this design can reach
The strongest form of this design is:

1. a much longer uninterrupted training horizon
2. larger and richer data
3. a stronger ViT backbone
4. a stronger family classifier
5. multi-timescale JEPA objectives
6. a stable always-on runtime

Under those conditions, the attractor families can become more semantic, more predictive, and more resistant to obvious low-dimensional failure modes.